# Evaluator Module
The Evaluator module creates evaluation reports.

Reports contain evaluation metrics depending on models specified in the evaluation config.

In [1]:
# reloads modules automatically before entering the execution of code
%load_ext autoreload
%autoreload 2

# third parties imports
import numpy as np 
import pandas as pd
# -- add new imports here --

# local imports
from configs import EvalConfig
from constants import Constant as C
from loaders import export_evaluation_report
from loaders import load_ratings
# -- add new imports here --
from surprise import accuracy
from surprise.model_selection import cross_validate, train_test_split, LeaveOneOut
from models import get_top_n  
import random as rd

# 1. Try the loader with surprise_format set to True
data = load_ratings(surprise_format=True)

# 2. Verify the results
print(f"Object Type: {type(data)}")

# 3. Build a trainset to confirm data is correctly loaded (check)
trainset = data.build_full_trainset()
print(f"Number of ratings: {trainset.n_ratings}")
print(f"Number of users: {trainset.n_users}")
print(f"Number of items: {trainset.n_items}")

print("Loader successfully tested in Surprise format!")


Object Type: <class 'surprise.dataset.DatasetAutoFolds'>
Number of ratings: 100004
Number of users: 671
Number of items: 9066
Loader successfully tested in Surprise format!


# 1. Model validation functions
Validation functions are a way to perform crossvalidation on recommender system models. 

In [2]:
def generate_split_predictions(algo, ratings_dataset, eval_config):
    """Generate predictions on a random test set specified in eval_config"""
    # Split the dataset into train and test sets using the size from eval_config
    trainset, testset = train_test_split(
        ratings_dataset, 
        test_size=eval_config.test_size, 
        random_state=42
    )
    
    # Train the algorithm on the training set
    algo.fit(trainset)
    
    # Generate predictions on the test set
    predictions = algo.test(testset)
    return predictions

def generate_loo_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user on a random Leave-one-out split (LOO)"""
    # Initialize the LeaveOneOut cross-validator
    loo = LeaveOneOut(n_splits=1, random_state=1)
    
    # Iterate through the split (only one iteration for n_splits=1)
    for trainset, testset in loo.split(ratings_dataset):
        # Train the algorithm on the training set
        algo.fit(trainset)
        
        # Build the anti-testset (all pairs of user-item NOT in the training set)
        anti_testset = trainset.build_anti_testset()
        
        # Generate predictions for the anti-testset
        predictions = algo.test(anti_testset)
        
        # Get top-n recommendations using the value from eval_config
        # We assign it to anti_testset_top_n as per the return requirement
        anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
        
    return anti_testset_top_n, testset


def generate_full_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user with full training set (LOO)"""
    # Build a training set using 100% of the available data
    full_trainset = ratings_dataset.build_full_trainset()
    
    # Train the algorithm on the full dataset
    algo.fit(full_trainset)
    
    # Build the anti-testset (all user-item pairs not present in the training data)
    anti_testset = full_trainset.build_anti_testset()
    
    # Generate predictions for the items users haven't seen yet
    predictions = algo.test(anti_testset)
    
    # Get top-n recommendations using the value from eval_config
    anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
    return anti_testset_top_n


#########################################
# Test of the three functions as required
#########################################
from models import ModelBaseline1
# from evaluation import generate_split_predictions, generate_loo_top_n, generate_full_top_n

# 1. Initialize the configuration instance

eval_config = EvalConfig()

# 2. Load the dataset in Surprise format
data = load_ratings(surprise_format=True)

# 3. Initialize the algorithm
algo = ModelBaseline1()

# --- (a) Test: Simple Train-Test Split ---
# Pass the instance 'eval_config' instead of the class 'EvalConfig'
predictions_a = generate_split_predictions(algo, data, eval_config)
print(f"Test (a): Success. {len(predictions_a)} raw predictions generated.")

# --- (b) Test: Leave-One-Out + Top-N ---
top_n_b, testset_b = generate_loo_top_n(algo, data, eval_config)
print(f"Test (b): Success. {len(top_n_b)} users with recommendations.")
for uid, recs in list(top_n_b.items())[:3]: 
    print(f"  User {uid}: {len(recs)} recommendations -> {recs}")

# --- (c) Test: Full Dataset Training + Top-N ---
top_n_c = generate_full_top_n(algo, data, eval_config)
print(f"Test (c): Success. {len(top_n_c)} users with recommendations.")
for uid, recs in list(top_n_c.items())[:3]:  
    print(f"  User {uid}: {len(recs)} recommendations -> {recs}")

# End of the test

def precompute_information(df_ratings):
    """ Returns a dictionary that precomputes relevant information for evaluating in full mode
    
    Dictionary keys:
    - precomputed_dict["item_to_rank"] : contains a dictionary mapping movie ids to rankings
    - (-- for your project, add other relevant information here -- )
    """
    precomputed_dict = {}
    
    # 1. Count how many times each movie appears in df_rating
    # movie_counts will have movie_id as index and frequency as value
    movie_counts = df_ratings[C.ITEM_ID_COL].value_counts()
    
    # 2. Create the ranking (Rank 1 = most frequent movie)
    # method='first' ensures we have integer ranks (1, 2, 3...) even in case of ties
    item_to_rank = movie_counts.rank(ascending=False, method='first').to_dict()
    
    # 3. Assign the result to the precomputed dictionary
    precomputed_dict["item_to_rank"] = item_to_rank

    return precomputed_dict                


def create_evaluation_report(eval_config, sp_ratings, precomputed_dict, available_metrics):
    """ Create a DataFrame evaluating various models on metrics specified in an evaluation config.  
    """
    evaluation_dict = {}
    for model_name, model, arguments in eval_config.models:
        print(f'Handling model {model_name}')
        algo = model(**arguments)
        evaluation_dict[model_name] = {}
        
        # Type 1 : split evaluations
        if len(eval_config.split_metrics) > 0:
            print('Training split predictions')
            predictions = generate_split_predictions(algo, sp_ratings, eval_config)
            for metric in eval_config.split_metrics:
                print(f'- computing metric {metric}')
                assert metric in available_metrics['split']
                evaluation_function, parameters =  available_metrics["split"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(predictions, **parameters) 

        # Type 2 : loo evaluations
        if len(eval_config.loo_metrics) > 0:
            print('Training loo predictions')
            anti_testset_top_n, testset = generate_loo_top_n(algo, sp_ratings, eval_config)
            for metric in eval_config.loo_metrics:
                assert metric in available_metrics['loo']
                evaluation_function, parameters =  available_metrics["loo"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(anti_testset_top_n, testset, **parameters)
        
        # Type 3 : full evaluations
        if len(eval_config.full_metrics) > 0:
            print('Training full predictions')
            anti_testset_top_n = generate_full_top_n(algo, sp_ratings, eval_config)
            for metric in eval_config.full_metrics:
                assert metric in available_metrics['full']
                evaluation_function, parameters =  available_metrics["full"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(
                    anti_testset_top_n,
                    **precomputed_dict,
                    **parameters
                )
        
    return pd.DataFrame.from_dict(evaluation_dict).T

Test (a): Success. 25001 raw predictions generated.
Test (b): Success. 671 users with recommendations.
  User 1: 40 recommendations -> [(102033, 2), (136864, 2), (104913, 2), (5570, 2), (5003, 2), (6057, 2), (2024, 2), (104283, 2), (132157, 2), (1673, 2), (4725, 2), (7361, 2), (8752, 2), (3861, 2), (3594, 2), (2440, 2), (7038, 2), (100326, 2), (4733, 2), (2759, 2), (3461, 2), (95508, 2), (4499, 2), (74787, 2), (7586, 2), (26228, 2), (50011, 2), (5825, 2), (328, 2), (6302, 2), (818, 2), (1150, 2), (167, 2), (114670, 2), (73106, 2), (5893, 2), (30749, 2), (2204, 2), (2479, 2), (95199, 2)]
  User 2: 40 recommendations -> [(5962, 2), (3177, 2), (37729, 2), (61250, 2), (4367, 2), (8609, 2), (103755, 2), (5475, 2), (190, 2), (7941, 2), (56176, 2), (6133, 2), (4511, 2), (7046, 2), (96314, 2), (2541, 2), (4933, 2), (3420, 2), (129653, 2), (78101, 2), (3543, 2), (6587, 2), (95115, 2), (3174, 2), (1454, 2), (101947, 2), (1292, 2), (6184, 2), (1994, 2), (2951, 2), (113862, 2), (67087, 2), (200, 2

# 2. Evaluation metrics
Implement evaluation metrics for either rating predictions (split metrics) or for top-n recommendations (loo metric, full metric)

In [3]:
def get_hit_rate(anti_testset_top_n, testset):
    """Compute the average hit over the users (loo metric)
    
    A hit (1) happens when the movie in the testset has been picked by the top-n recommender
    A fail (0) happens when the movie in the testset has not been picked by the top-n recommender
    """
    # -- implement the function get_hit_rate --
    hits = 0
    total_users = len(testset)

    # Iterate through each record in the testset (user, item, actual_rating)
    for user_id, movie_id, _ in testset:
        # Check if the user has recommendations in our top-n dictionary
        if user_id in anti_testset_top_n:
            # Get the list of recommended movie IDs for this user
            # We assume the top_n contains tuples (movie_id, estimated_rating)
            recommendations = [item_id for (item_id, _) in anti_testset_top_n[user_id]]
            
            # If the "hidden" movie from the testset is in the top-n list, it's a hit
            if movie_id in recommendations:
                hits += 1

    # Calculate the average hit rate
    hit_rate = hits / total_users if total_users > 0 else 0
    
    return hit_rate


def get_novelty(anti_testset_top_n, item_to_rank):
    """Compute the average novelty of the top-n recommendation over the users (full metric)
    
    The novelty is defined as the average ranking of the movies recommended
    """
    # -- implement the function get_novelty --
    total_novelty = 0
    total_users = len(anti_testset_top_n)

    for user_id, recommendations in anti_testset_top_n.items():
        user_novelty_sum = 0
        n_items = len(recommendations)
        
        for movie_id, _ in recommendations:
            # We look up the popularity rank of the movie
            # If the movie isn't in our rank dict, we could default to a high rank
            rank = item_to_rank.get(movie_id, len(item_to_rank))
            user_novelty_sum += rank
        
        # We average the sum of ranks for this user's top-n list
        # Then add it to the total to get the global average later
        if n_items > 0:
            total_novelty += user_novelty_sum

    # Compute the average novelty across all users
    average_rank_sum = total_novelty / total_users if total_users > 0 else 0
    
    return average_rank_sum

## Pitfalls of Using a Sum of Ranks as a Novelty Metric

1. **Sensitivity to N (top-n size)**: If two models recommend different numbers
   of items (e.g. 10 vs 40), the sum of ranks will be mechanically higher for
   the one recommending more items, without being truly more "novel".

2. **Linearity of ranks**: The difference between rank 1 and rank 100 is
   treated the same as between rank 1000 and rank 1100. However, moving from
   the most popular movie to the 100th is a far more radical shift in popularity
   than moving from rank 1000 to 1100. A logarithmic scale would better capture
   this reality.

3. **Quality vs. Novelty trade-off**: A model could achieve a very high novelty
   score by recommending the least popular (and potentially worst) movies on the
   platform that nobody wants to watch. Novelty should always be balanced with
   precision metrics (MAE, RMSE, Hit Rate) to ensure recommendations remain
   relevant.

4. **Dependence on catalogue size**: A rank of 500 in a catalogue of 600 movies
   does not have the same meaning as a rank of 500 in a catalogue of 100,000
   movies. The metric is therefore not comparable across systems with catalogues
   of different sizes.

# 3. Evaluation workflow
Load data, evaluate models and save the experimental outcomes

In [4]:
np.random.seed(1)
rd.seed(1)

AVAILABLE_METRICS = {
    "split": {
        "mae": (accuracy.mae, {'verbose': False}),
        # -- add new split metrics here --
        "rmse": (accuracy.rmse, {'verbose': False})
    },
    # -- add new types of metrics here --
    "loo": {
        "hit_rate": (get_hit_rate, {}) 
    },
    "full": {
        "novelty": (get_novelty, {})   
    }
}

df_ratings_pd = load_ratings(surprise_format=False)
sp_ratings = load_ratings(surprise_format=True)
precomputed_dict = precompute_information(df_ratings_pd)
evaluation_report = create_evaluation_report(EvalConfig(), sp_ratings, precomputed_dict, AVAILABLE_METRICS)
export_evaluation_report(evaluation_report)
display(evaluation_report)

Handling model KNNwithMeans
Training split predictions
Computing the msd similarity matrix...
Done computing similarity matrix.
- computing metric rmse
Handling model UserBased_Manual
Training split predictions
- computing metric rmse
Handling model UserBased_tuned
Training split predictions
- computing metric rmse
Handling model RandomScore
Training split predictions
- computing metric rmse


,rmse
KNNwithMeans,0.993923
UserBased_Manual,0.985767
UserBased_tuned,0.924767
RandomScore,1.818705


## Evaluation Report Observations

**Note**: Results obtained on the test dataset (6 users, 10 items),
used only to verify that the pipeline is working correctly.

### 1. Precision Metrics (MAE & RMSE)
- **Baseline 4 (SVD)** is the best performing model on MAE (0.954). This is
  consistent since SVD is a learning algorithm that minimizes prediction error,
  unlike the static baselines.
- **Baseline 1** is the least performant (MAE=1.125): always predicting the same
  value captures none of the nuances of user preferences.
- RMSE is systematically higher than MAE for all models, which is expected:
  RMSE penalizes large errors more heavily (squared differences).

### 2. Hit Rate (1.0)
- All 4 models display a Hit Rate of 1.0 (100%), which would be impossible
  in a real scenario.
- This is explained by the dataset size: with only ~10 available movies and
  a top_n_value=40, the model will always include the "hidden" movie in its list.
- The metric is correctly implemented, but will only be discriminant on a larger
  dataset (thousands of movies, with only 40 possible recommendations).

### 3. Novelty (~30.17)
- The value is nearly identical across all baselines.
- For models predicting constant or average values (Baseline 1 and 3), the
  ordering of recommendations depends on item appearance order or tie-breaking.
- On such a small dataset, all models end up recommending every available movie,
  making the average popularity rank mechanically identical for everyone.

## Content-Based Models Evaluation

### 1. Random Sample vs. Random Score
Random Sample (RMSE 1.31) outperforms Random Score (RMSE 1.79). Random Sample draws from the user's own rating distribution, centered around their personal mean (~3.5 on MovieLens), whereas Random Score samples uniformly in [0.5, 5] (mean ~2.75). Predictions from Random Sample are therefore structurally closer to real ratings.

### 2. Linear Regression (Intercept True vs. False)
Linear Regression with `fit_intercept=True` (RMSE 0.93) vastly outperforms `fit_intercept=False` (RMSE 1.55). Without an intercept, the model must explain a ~3.5-star average using only `coef × title_length`, which is impossible without distorting the coefficient. The intercept captures the user's baseline rating tendency — essentially their mean. With a single weak feature like title length, the intercept does almost all the work: 0.93 is roughly what a "predict the user's mean" baseline would achieve.

### 3. Comparing more advanced models
As base features, we used `all_content_tmdb_tags2000`, which concatenates genome scores (1128 dims), a rich TF-IDF on user tags (up to 2000 features, bigrams, sublinear TF), normalized release year + decade one-hot, genres (TF-IDF), and TMDB metadata (runtime, language, country, studio, directors, cast, keywords, collection, budget, writers, release date, overview TF-IDF). The total feature space spans several thousand dimensions. We compared two regressors:
- **Ridge**: L2-regularized linear regression with a fixed `alpha=1.0` for all users.
- **RidgeCV**: same model, but `alpha` is selected per user via cross-validation over `1e-4` to `1e5`.

**RidgeCV (0.74) outperforms standard Ridge (0.93)** because with several thousand features and only tens to hundreds of ratings per user, `alpha=1.0` is severely under-regularized: Ridge overfits and falls back to roughly the same RMSE as a "predict the user's mean" model. RidgeCV picks larger alphas for sparse profiles (strong shrinkage) and smaller ones for rich profiles, adapting regularization to each user's data volume. Notably, **RidgeCV (0.744) even beats the best collaborative baseline so far, SVD (0.817)**, using only item content and per-user ridge profiles.